In [ ]:
import csv
import pandas as pd
import numpy as np
import scanpy as sc

import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
import seaborn as sns

import scipy as sp
from scipy import stats as st
import statsmodels.stats.anova as anova 
from statsmodels.stats.multicomp import pairwise_tukeyhsd

import yaml
from pathlib import Path

from collections import defaultdict

In [ ]:
sc.set_figure_params(dpi=100, color_map = 'viridis_r')
sc.settings.verbosity = 1
sc.logging.print_header()

In [ ]:
adata = sc.read("/data/projects/manuela/scRNA_aging-mouse/integrated-aging-soupxed.h5ad", var_names="gene_symbols")

In [ ]:
split_adatas = [
    adata[adata.obs['sample'] == sample].copy() for sample in adata.obs['sample'].unique()
]
split_adatas[1].obs

In [ ]:
sc.tl.dendrogram(adata, groupby='sample')
sc.pl.dendrogram(adata, 'sample')
print(adata.uns['dendrogram_sample'])
sc.pl.correlation_matrix(adata, 'sample')

# Plot UMAP

In [ ]:
# compute clusters using the leiden method and store the results with the name `clusters`
sc.tl.leiden(adata, key_added='clusters', resolution=0.31)
#sc.tl.umap(adata)

In [ ]:
#rc_context is used for the figure size, in this case 4x4
adata.obs.clusters = adata.obs.clusters.astype('category')
with rc_context({'figure.figsize': (4, 4)}):
    sc.pl.umap(adata, color=['sample', 'clusters',], wspace=0.4, save = 'Umap_aging.png')    
    sc.pl.umap(adata, color=['n_genes_by_counts', 'total_counts', 'nFeature_RNA'], wspace=0.4)   

## Calculate Cell Counts per Cluster & per Sample per Cluster

In [ ]:
#Per cluster
counts = adata.obs['clusters'].value_counts()

fig, ax = plt.subplots(figsize=(10, 6))
counts.plot(kind="bar", ax=ax)

# Add labels and title
plt.xlabel('Cluster Label')
plt.ylabel('Count')
plt.title('Cell Counts per Cluster')

# Show the plot
plt.show()

In [ ]:
#Per Sample
counts = adata.obs['sample'].value_counts()

fig, ax = plt.subplots(figsize=(10, 6))
counts.plot(kind="bar", ax=ax)

# Add labels and title
plt.xlabel('Cluster Label')
plt.ylabel('Count')
plt.title('Cell Counts per Sample')

# Show the plot
plt.show()

In [ ]:
# create counts_per_sample dataframe
counts_per_sample = adata.obs.groupby(['sample', 'clusters']).size().unstack(fill_value=0)

# plot heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(counts_per_sample, cmap="YlGnBu")
plt.title("Cell counts per sample per cluster")
plt.show()

# Create kidney Celltype Marker Gene List

In [ ]:
#import celltype marker list
celltypemarkers_dict = yaml.safe_load(Path('/data/projects/manuela/bulkRNA_mouse/mouse_markers_detailed.yaml').read_text())

# Loop over dictionary and remove newline from values
for key, value in celltypemarkers_dict.items():
    celltypemarkers_dict[key] = [v.replace('\n', '') for v in value]
    
#subset
subset_dict = {k: [i for i in v if i in adata.raw.var_names.values.tolist()] for k, v in celltypemarkers_dict.items() if any(i in v for i in adata.raw.var_names.values.tolist())}            
subset_dict

In [ ]:
# create list with all celltypemarker genes
celltypemarkers_list_notflat = celltypemarkers_dict.values()
celltypemarkers_list = [item for sublist in celltypemarkers_list_notflat for item in sublist]
#print(celltypemarkers_list)

In [ ]:
#Filter celltypemarkers -> use only the ones that are expressed
celltypemarkers_sub = [item for item in celltypemarkers_list if item in adata.raw.var_names]
print(len(celltypemarkers_list))
print(len(celltypemarkers_sub))
#celltypemarkers_sub

In [ ]:
# Dotplot with celltypemarkers from marker gene list
sc.pl.DotPlot(adata, use_raw=True, var_names=celltypemarkers_sub, groupby='clusters', title='Celltypemarkers genes expression in clusters ', cmap='BuGn').savefig('dotplot_agingMouse.png')

In [ ]:
# Dotplot with celltypemarkers and celltypes from marker gene disctionary
sc.pl.DotPlot(adata, use_raw=True, var_names=subset_dict, groupby='clusters', standard_scale="var", title='Celltypemarkers genes expression in clusters ', cmap='BuGn').savefig('dotplot_agingMouse.png')

# DE Analysis by Clusters

In [ ]:
adata.uns['log1p']["base"] = None
sc.tl.rank_genes_groups(adata, "clusters", method='t-test')
#sc.pl.rank_genes_groups(adata, n_genes=10, sharey=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(
    adata, groupby="clusters", standard_scale="var", n_genes=5, title='Top 5 Genes expression in clusters', key="rank_genes_groups"
)

## Intersection of Top 10/20 DEGs per Cluster with Reference Marker List

In [ ]:
# Get Top DE genes per cluster
result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names
top_genes_per_cluster = {}
for group in groups:
    top_genes_per_cluster[group] = result['names'][group][:50]

# Get all unique genes across all clusters
all_top_genes = set().union(*top_genes_per_cluster.values())
#all_top_genes

In [ ]:
#convert reference marker genes-> cell types dictionary to dataframe 
reference_genes_df = pd.DataFrame(list(celltypemarkers_dict.items()), columns=['cell_type', 'marker_genes'])
reference_genes_df = reference_genes_df.explode('marker_genes')
# Reset the index of the dataframe
reference_genes_df = reference_genes_df.reset_index(drop=True)
reference_genes_df

In [ ]:
reference_genes_df.to_excel("reference_genes_df.xlsx") 

In [ ]:
# Intersection of the reference markers and the top 10 genes per cluster
reference_genes = set(reference_genes_df['marker_genes'])
intersection_genes = reference_genes.intersection(all_top_genes)

# Subset the reference markers dictionary to keep only the genes in the intersection
reference_markers_subset = {gene: markers for markers, genes in celltypemarkers_dict.items() for gene in genes if gene in intersection_genes}
#reference_markers_subset

In [ ]:
celltypemarkers_dict_subset = {}
for marker, genes in celltypemarkers_dict.items():
    subset_genes = [gene for gene in genes if gene in intersection_genes]
    #print(subset_genes)
    if len(subset_genes) > 0:
        celltypemarkers_dict_subset[marker] = subset_genes
#celltypemarkers_dict_subset

### Dotplot of Intersected DEGs from Reference Marker List 

In [ ]:
# Dotplot with celltypemarkers and celltypes from marker gene disctionary
sc.pl.DotPlot(adata, use_raw=True, var_names=celltypemarkers_dict_subset, groupby='clusters', title='Celltypemarkers genes expression in clusters ', cmap='BuGn').savefig('dotplot_agingMouse.png')

In [ ]:
sc.pl.rank_genes_groups_dotplot(
    adata, groupby="clusters", standard_scale="var", var_names=celltypemarkers_dict_subset, title='Top 5 Genes expression in clusters', key="rank_genes_groups"
)

In [ ]:
celltypes_sub = ['POD', 'PT', 'EC', 'TAL', 'LEUK']
celltypemarkers_dict_sub = {k: v for k, v in celltypemarkers_dict_subset.items() if k in celltypes_sub}

sc.pl.rank_genes_groups_dotplot(
    adata, groupby="clusters", standard_scale="var", var_names=celltypemarkers_dict_sub, title='Top 5 Genes expression in clusters', key="rank_genes_groups"
)

In [ ]:
sc.pl.rank_genes_groups_dotplot(
    adata, groupby="clusters", standard_scale="var", var_names=celltypemarkers_dict_subset, title='Top 5 Genes expression in clusters', key="rank_genes_groups"
)

In [ ]:
sc.pl.matrixplot(adata, celltypemarkers_dict_subset, 'clusters', dendrogram=True, cmap='Blues', standard_scale='var', colorbar_title='column scaled\nexpression')

## Excel Tables

In [ ]:
cluster_gene_table = sc.get.rank_genes_groups_df(adata, group= None)
cluster_gene_table

In [ ]:
df = pd.pivot_table(data=cluster_gene_table, index=['names'], columns=['group'])
df

In [ ]:
with pd.ExcelWriter("AgingMouse_ClusterMarkers.xlsx", engine='openpyxl') as writer:
    for clustername in df.columns.levels[1]:
        df.xs(clustername, level="group", axis=1).to_excel(writer, sheet_name=f"Cluster_{clustername}_vs. rest")